# Ensemble Baseline + V3 - Cassava Disease Classification

## 🎯 Objectif
Combiner les forces des deux modèles:
- **Baseline**: Expert sur CBSD (49%), CMD (97%), Healthy (46%)
- **V3**: Expert sur CBB (34%), CGM (81%)

## 📊 Performance Attendue
- Overall: **~70-73%** (meilleur possible!)
- CBB: ~34% (de V3)
- CBSD: ~49% (de baseline)
- CGM: ~81% (de V3)
- CMD: ~97% (de baseline)
- Healthy: ~46% (de baseline)

## 📝 Stratégie de Routage (V2 - Basé sur Confiance)

**Stratégie Simple**:
1. Exécuter les deux modèles sur chaque image
2. Comparer les scores de confiance (= accuracy connue par classe)
3. Utiliser la prédiction du modèle avec la **PLUS HAUTE confiance**

**Pourquoi ça fonctionne**:
- CBSD: baseline (49%) > V3 (3%) → utilise baseline ✅
- CMD: baseline (97%) > V3 (89%) → utilise baseline ✅
- Healthy: baseline (46%) > V3 (40%) → utilise baseline ✅
- CBB: V3 (34%) > baseline (0.6%) → utilise V3 ✅
- CGM: V3 (81%) > baseline (0%) → utilise V3 ✅

**Note**: La V1 (routage par classe) a échoué car elle routait basé sur la *prédiction* au lieu de la *confiance*.

---

## 1. Imports et Configuration

In [1]:
import os
import json
import re
import random
import numpy as np
import torch
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from collections import Counter

from unsloth import FastVisionModel
from transformers import AutoProcessor

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch version: 2.9.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5090


## 2. Configuration des Chemins

In [2]:
# Chemins
BASE_DIR = Path("/home/hounfodjidagba/learning/cassava/new_implementation")
DATA_DIR = BASE_DIR / "data" / "stage1"

# Checkpoints
BASELINE_CHECKPOINT = BASE_DIR / "checkpoints" / "stage1_cassava_gemma3n" / "checkpoint-???"  # TODO: trouver le meilleur
V3_CHECKPOINT = BASE_DIR / "checkpoints" / "stage1_cassava_gemma3n_v3" / "final_model"

# Classes
CLASS_NAMES = [
    "Cassava Bacterial Blight (CBB)",
    "Cassava Brown Streak Disease (CBSD)",
    "Cassava Green Mottle (CGM)",
    "Cassava Mosaic Disease (CMD)",
    "Healthy"
]

# Accuracies connues (utilisées comme scores de confiance)
BASELINE_ACCURACY = {
    'Cassava Bacterial Blight (CBB)': 0.0064,  # 0.64%
    'Cassava Brown Streak Disease (CBSD)': 0.4864,  # 48.64%
    'Cassava Green Mottle (CGM)': 0.0,  # 0%
    'Cassava Mosaic Disease (CMD)': 0.9707,  # 97.07%
    'Healthy': 0.4577  # 45.77%
}

V3_ACCURACY = {
    'Cassava Bacterial Blight (CBB)': 0.34,  # 34%
    'Cassava Brown Streak Disease (CBSD)': 0.029,  # 2.9%
    'Cassava Green Mottle (CGM)': 0.814,  # 81.4%
    'Cassava Mosaic Disease (CMD)': 0.894,  # 89.4%
    'Healthy': 0.403  # 40.3%
}

print("✅ Configuration chargée")

✅ Configuration chargée


## 3. Trouver le Meilleur Checkpoint Baseline

In [3]:
# Lister les checkpoints disponibles
baseline_dir = BASE_DIR / "checkpoints" / "stage1_cassava_gemma3n"

if baseline_dir.exists():
    checkpoints = list(baseline_dir.glob("checkpoint-*"))
    print(f"Checkpoints baseline trouvés: {len(checkpoints)}")
    
    # Chercher celui avec le meilleur eval_loss
    best_checkpoint = None
    best_eval_loss = float('inf')
    
    for ckpt in checkpoints:
        trainer_state = ckpt / "trainer_state.json"
        if trainer_state.exists():
            with open(trainer_state, 'r') as f:
                state = json.load(f)
                # Chercher le eval_loss le plus récent
                for log in reversed(state.get('log_history', [])):
                    if 'eval_loss' in log:
                        eval_loss = log['eval_loss']
                        print(f"  {ckpt.name}: eval_loss={eval_loss:.4f}")
                        if eval_loss < best_eval_loss:
                            best_eval_loss = eval_loss
                            best_checkpoint = ckpt
                        break
    
    if best_checkpoint:
        BASELINE_CHECKPOINT = best_checkpoint
        print(f"\n✅ Meilleur checkpoint baseline: {BASELINE_CHECKPOINT.name}")
        print(f"   eval_loss: {best_eval_loss:.4f}")
    else:
        # Utiliser final_model si disponible
        final_model = baseline_dir / "final_model"
        if final_model.exists():
            BASELINE_CHECKPOINT = final_model
            print(f"✅ Utilisation de final_model")
else:
    print(f"⚠️ Répertoire baseline non trouvé: {baseline_dir}")
    print("   Vérifiez le chemin ou ajustez manuellement BASELINE_CHECKPOINT")

Checkpoints baseline trouvés: 4
  checkpoint-8500: eval_loss=6.1863
  checkpoint-3500: eval_loss=5.2267
  checkpoint-8000: eval_loss=5.6148

✅ Meilleur checkpoint baseline: checkpoint-3500
   eval_loss: 5.2267


## 4. Chargement des Modèles

In [4]:
print("="*60)
print("CHARGEMENT DES MODÈLES")
print("="*60)

# Charger Baseline
print("\n[1/2] Chargement Baseline...")
baseline_model, baseline_tokenizer = FastVisionModel.from_pretrained(
    str(BASELINE_CHECKPOINT),
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth"
)
baseline_processor = AutoProcessor.from_pretrained(str(BASELINE_CHECKPOINT))
FastVisionModel.for_inference(baseline_model)
print("✅ Baseline chargé")

# Charger V3
print("\n[2/2] Chargement V3...")
v3_model, v3_tokenizer = FastVisionModel.from_pretrained(
    str(V3_CHECKPOINT),
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth"
)
v3_processor = AutoProcessor.from_pretrained(str(V3_CHECKPOINT))
FastVisionModel.for_inference(v3_model)
print("✅ V3 chargé")

print(f"\n{'='*60}")
print("✅ Les deux modèles sont prêts")
print(f"{'='*60}")

CHARGEMENT DES MODÈLES

[1/2] Chargement Baseline...


==((====))==  Unsloth 2026.1.1: Fast Gemma3N patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

✅ Baseline chargé

[2/2] Chargement V3...
==((====))==  Unsloth 2026.1.1: Fast Gemma3N patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

✅ V3 chargé

✅ Les deux modèles sont prêts


## 5. Fonctions d'Extraction de Classes

In [5]:
def extract_class_from_response(response: str, is_v3: bool = False) -> str:
    """
    Extrait la classe prédite de la réponse du modèle.
    
    Args:
        response: Réponse brute du modèle
        is_v3: Si True, utilise le format multiple-choice (A/B/C/D/E)
    
    Returns:
        Nom de la classe prédite
    """
    if is_v3:
        # V3 utilise le format A) B) C) D) E)
        response = response.strip()
        
        # Nettoyer la réponse
        if "model\n" in response:
            response = response.split("model\n")[-1].strip()
        
        # Mapping lettre → classe
        letter_to_class = {
            'A': 'Cassava Bacterial Blight (CBB)',
            'B': 'Cassava Brown Streak Disease (CBSD)',
            'C': 'Cassava Green Mottle (CGM)',
            'D': 'Cassava Mosaic Disease (CMD)',
            'E': 'Healthy'
        }
        
        # Chercher la lettre au début
        for letter in ['A', 'B', 'C', 'D', 'E']:
            if response.startswith(f"{letter})") or response.startswith(f"{letter} )"):
                return letter_to_class[letter]
        
        # Fallback: chercher n'importe où
        match = re.search(r'\b([A-E])\)', response)
        if match:
            return letter_to_class[match.group(1)]
    
    # Baseline ou fallback: chercher les noms de maladies
    response_lower = response.lower()
    if 'bacterial blight' in response_lower or 'cbb' in response_lower:
        return 'Cassava Bacterial Blight (CBB)'
    elif 'brown streak' in response_lower or 'cbsd' in response_lower:
        return 'Cassava Brown Streak Disease (CBSD)'
    elif 'green mottle' in response_lower or 'cgm' in response_lower:
        return 'Cassava Green Mottle (CGM)'
    elif 'mosaic' in response_lower or 'cmd' in response_lower:
        return 'Cassava Mosaic Disease (CMD)'
    elif 'healthy' in response_lower:
        return 'Healthy'
    
    return "Unknown"

# Test rapide
test_cases = [
    ("' and full name only.\nmodel\nD) Cassava Mosaic Disease (CMD)", True, "CMD"),
    ("The disease shown is Cassava Mosaic Disease (CMD)", False, "CMD"),
    ("A) Cassava Bacterial Blight (CBB)", True, "CBB")
]

print("Test de la fonction d'extraction:")
for response, is_v3, expected in test_cases:
    result = extract_class_from_response(response, is_v3)
    status = "✅" if expected in result else "❌"
    print(f"  {status} '{response[:40]}...' → {result}")

print("\n✅ Fonction d'extraction définie")

Test de la fonction d'extraction:
  ✅ '' and full name only.
model
D) Cassava M...' → Cassava Mosaic Disease (CMD)
  ✅ 'The disease shown is Cassava Mosaic Dise...' → Cassava Mosaic Disease (CMD)
  ✅ 'A) Cassava Bacterial Blight (CBB)...' → Cassava Bacterial Blight (CBB)

✅ Fonction d'extraction définie


## 6. Fonctions d'Inférence

In [6]:
def predict_with_baseline(image: Image.Image):
    """
    Prédiction avec le modèle baseline.
    
    Returns:
        (class_name, confidence, raw_response)
    """
    # Prompt open-ended pour baseline
    prompt = "What cassava disease is shown in this image?"
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt}
            ]
        }
    ]
    
    input_text = baseline_processor.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    inputs = baseline_processor(
        text=[input_text],
        images=[[image]],
        return_tensors="pt",
        padding=True
    ).to(baseline_model.device)
    
    with torch.no_grad():
        outputs = baseline_model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            temperature=None,
            top_p=None
        )
    
    response = baseline_processor.tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extraire la classe
    predicted_class = extract_class_from_response(response, is_v3=False)
    
    # Confiance = accuracy connue
    confidence = BASELINE_ACCURACY.get(predicted_class, 0.5)
    
    return predicted_class, confidence, response


def predict_with_v3(image: Image.Image):
    """
    Prédiction avec le modèle V3.
    
    Returns:
        (class_name, confidence, raw_response)
    """
    # Prompt multiple-choice pour V3
    prompt = """Look at this cassava leaf image and identify the disease.

Choose EXACTLY ONE option:
A) Cassava Bacterial Blight (CBB)
B) Cassava Brown Streak Disease (CBSD)
C) Cassava Green Mottle (CGM)
D) Cassava Mosaic Disease (CMD)
E) Healthy

Answer with the letter and full name only."""
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt}
            ]
        }
    ]
    
    input_text = v3_processor.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    inputs = v3_processor(
        text=[input_text],
        images=[[image]],
        return_tensors="pt",
        padding=True
    ).to(v3_model.device)
    
    with torch.no_grad():
        outputs = v3_model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False,
            temperature=None,
            top_p=None
        )
    
    response = v3_processor.tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extraire la classe
    predicted_class = extract_class_from_response(response, is_v3=True)
    
    # Confiance = accuracy connue
    confidence = V3_ACCURACY.get(predicted_class, 0.5)
    
    return predicted_class, confidence, response

print("✅ Fonctions d'inférence définies")

✅ Fonctions d'inférence définies


## 7. Logique d'Ensemble - V2 (Routage par Confiance)

**🔧 CORRECTION**: La version V1 utilisait un routage par classe qui causait des erreurs.

**Problème V1**: 
- Routait basé sur ce que V3 **prédisait** (ex: si V3 prédit CGM, utilise V3)
- Mais V3 sur-prédit CGM → CBSD samples mal classifiés → 0% CBSD accuracy!

**Solution V2**:
- Routage basé uniquement sur la **CONFIANCE** (= accuracy connue par classe)
- Exemple: Pour CBSD, baseline (49%) > V3 (3%) → utilise baseline
- Simple et efficace!

In [7]:
def ensemble_predict(image: Image.Image):
    """
    V2: Prédiction d'ensemble avec routage par CONFIANCE pure.

    🔧 FIX: Remplace le routage par classe (qui routait sur la PRÉDICTION)
           par un routage basé uniquement sur la CONFIANCE (accuracy connue).

    Stratégie:
    - Simplement utiliser le modèle avec la PLUS HAUTE confiance
    - Pas de routage par classe (ça causait des erreurs quand V3 prédisait mal)

    Returns:
        Dict avec tous les détails de la prédiction
    """
    # Obtenir les prédictions des deux modèles
    baseline_class, baseline_conf, baseline_raw = predict_with_baseline(image)
    v3_class, v3_conf, v3_raw = predict_with_v3(image)

    # STRATÉGIE SIMPLE: Utiliser le modèle avec la plus haute confiance
    # Les confiances sont basées sur les accuracies connues par classe:
    # - CBSD: baseline (0.49) > V3 (0.03) → baseline gagne
    # - CMD:  baseline (0.97) > V3 (0.89) → baseline gagne
    # - CBB:  V3 (0.34) > baseline (0.006) → V3 gagne
    # - CGM:  V3 (0.81) > baseline (0.0) → V3 gagne
    # - Healthy: baseline (0.46) > V3 (0.40) → baseline gagne

    if baseline_conf >= v3_conf:
        final_class = baseline_class
        final_conf = baseline_conf
        source = 'baseline'
    else:
        final_class = v3_class
        final_conf = v3_conf
        source = 'v3'

    return {
        'predicted_class': final_class,
        'confidence': final_conf,
        'source_model': source,
        'baseline_prediction': baseline_class,
        'baseline_confidence': baseline_conf,
        'v3_prediction': v3_class,
        'v3_confidence': v3_conf,
        'baseline_raw': baseline_raw,
        'v3_raw': v3_raw,
        'agreement': baseline_class == v3_class
    }

print("✅ Logique d'ensemble V2 définie (routage par CONFIANCE)")
print(f"\nStratégie:")
print(f"  - Utilise le modèle avec la PLUS HAUTE confiance")
print(f"  - Confiance = accuracy connue par classe")
print(f"  - Pas de routage par classe (causait des erreurs)")
print(f"\nExemples:")
print(f"  - CBSD: baseline (49%) > V3 (3%) → utilise baseline")
print(f"  - CMD:  baseline (97%) > V3 (89%) → utilise baseline")
print(f"  - CBB:  V3 (34%) > baseline (0.6%) → utilise V3")
print(f"  - CGM:  V3 (81%) > baseline (0%) → utilise V3")

✅ Logique d'ensemble V2 définie (routage par CONFIANCE)

Stratégie:
  - Utilise le modèle avec la PLUS HAUTE confiance
  - Confiance = accuracy connue par classe
  - Pas de routage par classe (causait des erreurs)

Exemples:
  - CBSD: baseline (49%) > V3 (3%) → utilise baseline
  - CMD:  baseline (97%) > V3 (89%) → utilise baseline
  - CBB:  V3 (34%) > baseline (0.6%) → utilise V3
  - CGM:  V3 (81%) > baseline (0%) → utilise V3


## 8. Chargement du Test Set

In [8]:
# Charger le test set
test_file = DATA_DIR / "cassava_test.json"
with open(test_file, 'r') as f:
    test_data = json.load(f)

print(f"✅ Test set chargé: {len(test_data)} échantillons")

# Fonction pour extraire la classe ground truth
def extract_gt_class(sample):
    """Extrait la classe ground truth d'un échantillon."""
    conversations = sample.get('conversations', [])
    for turn in conversations:
        if turn.get('from') == 'assistant' or turn.get('role') == 'assistant':
            content = turn.get('value') or turn.get('content', '')
            
            if 'Bacterial Blight' in content or 'CBB' in content:
                return 'Cassava Bacterial Blight (CBB)'
            elif 'Brown Streak' in content or 'CBSD' in content:
                return 'Cassava Brown Streak Disease (CBSD)'
            elif 'Green Mottle' in content or 'CGM' in content:
                return 'Cassava Green Mottle (CGM)'
            elif 'Mosaic' in content or 'CMD' in content:
                return 'Cassava Mosaic Disease (CMD)'
            elif 'Healthy' in content:
                return 'Healthy'
    return 'Unknown'

# Fonction pour extraire le chemin de l'image
def extract_image_path(sample):
    """Extrait le chemin de l'image d'un échantillon."""
    conversations = sample.get('conversations', [])
    for turn in conversations:
        if turn.get('from') == 'user' or turn.get('role') == 'user':
            content = turn.get('value') or turn.get('content', '')
            match = re.search(r'<img>(.*?)</img>', content)
            if match:
                return match.group(1)
    return None

# Analyser la distribution
gt_distribution = Counter()
for sample in test_data:
    gt_class = extract_gt_class(sample)
    gt_distribution[gt_class] += 1

print(f"\nDistribution du test set:")
for cls, count in gt_distribution.most_common():
    pct = count / len(test_data) * 100
    print(f"  {cls}: {count} ({pct:.1f}%)")

✅ Test set chargé: 3619 échantillons

Distribution du test set:
  Cassava Mosaic Disease (CMD): 2082 (57.5%)
  Cassava Brown Streak Disease (CBSD): 662 (18.3%)
  Healthy: 402 (11.1%)
  Cassava Green Mottle (CGM): 317 (8.8%)
  Cassava Bacterial Blight (CBB): 156 (4.3%)


## 9. Test Rapide (100 échantillons)

In [9]:
# Sélectionner 100 échantillons aléatoires
random.seed(42)
quick_test_samples = random.sample(test_data, 100)

print("="*70)
print("TEST RAPIDE - 100 ÉCHANTILLONS")
print("="*70)

results = []
errors = []

for sample in tqdm(quick_test_samples, desc="Prédictions"):
    gt_class = extract_gt_class(sample)
    image_path = extract_image_path(sample)
    
    if not image_path or gt_class == 'Unknown':
        continue
    
    try:
        # Charger l'image
        image = Image.open(image_path).convert('RGB')
        
        # Prédiction d'ensemble
        prediction = ensemble_predict(image)
        
        is_correct = prediction['predicted_class'] == gt_class
        
        results.append({
            'id': sample.get('id', 'unknown'),
            'ground_truth': gt_class,
            'predicted': prediction['predicted_class'],
            'confidence': prediction['confidence'],
            'source': prediction['source_model'],
            'baseline_pred': prediction['baseline_prediction'],
            'v3_pred': prediction['v3_prediction'],
            'correct': is_correct
        })
        
        if not is_correct:
            errors.append({
                'id': sample.get('id', 'unknown'),
                'true': gt_class,
                'predicted': prediction['predicted_class'],
                'source': prediction['source_model']
            })
    
    except Exception as e:
        print(f"\nErreur: {e}")
        continue

print(f"\n✅ Test rapide terminé: {len(results)} prédictions")

TEST RAPIDE - 100 ÉCHANTILLONS


Prédictions: 100%|██████████| 100/100 [11:40<00:00,  7.00s/it]


✅ Test rapide terminé: 100 prédictions


## 10. Analyse des Résultats (Quick Test)

In [10]:
# Calculer les métriques
total = len(results)
correct = sum(1 for r in results if r['correct'])
overall_acc = correct / total if total > 0 else 0

print("="*70)
print("RÉSULTATS - TEST RAPIDE (100 échantillons)")
print("="*70)

print(f"\n📊 OVERALL:")
print(f"   Accuracy: {overall_acc*100:.2f}% ({correct}/{total})")

# Par classe
print(f"\n📈 PAR CLASSE:")
class_metrics = {}
for class_name in CLASS_NAMES:
    class_results = [r for r in results if r['ground_truth'] == class_name]
    if class_results:
        class_correct = sum(1 for r in class_results if r['correct'])
        class_acc = class_correct / len(class_results)
        class_metrics[class_name] = {
            'total': len(class_results),
            'correct': class_correct,
            'accuracy': class_acc
        }
        
        status = "✅" if class_acc >= 0.30 else "⚠️" if class_acc >= 0.20 else "❌"
        abbrev = class_name.split('(')[1].strip(')') if '(' in class_name else class_name
        print(f"   {status} {abbrev}: {class_acc*100:.1f}% ({class_correct}/{len(class_results)})")

# Utilisation des modèles
baseline_count = sum(1 for r in results if r['source'] == 'baseline')
v3_count = sum(1 for r in results if r['source'] == 'v3')

print(f"\n🤖 UTILISATION DES MODÈLES:")
print(f"   Baseline: {baseline_count} ({baseline_count/total*100:.1f}%)")
print(f"   V3: {v3_count} ({v3_count/total*100:.1f}%)")

# Accord entre modèles
agreements = sum(1 for r in results if r['baseline_pred'] == r['v3_pred'])
print(f"   Accord: {agreements}/{total} ({agreements/total*100:.1f}%)")

# Erreurs
if errors:
    print(f"\n❌ ERREURS (premières 10):")
    for i, err in enumerate(errors[:10], 1):
        true_abbrev = err['true'].split('(')[1].strip(')') if '(' in err['true'] else err['true']
        pred_abbrev = err['predicted'].split('(')[1].strip(')') if '(' in err['predicted'] else err['predicted']
        print(f"   {i}. {true_abbrev} → {pred_abbrev} (via {err['source']})")

print("\n" + "="*70)

RÉSULTATS - TEST RAPIDE (100 échantillons)

📊 OVERALL:
   Accuracy: 50.00% (50/100)

📈 PAR CLASSE:
   ❌ CBB: 0.0% (0/5)
   ❌ CBSD: 0.0% (0/21)
   ✅ CGM: 75.0% (6/8)
   ✅ CMD: 78.6% (44/56)
   ❌ Healthy: 0.0% (0/10)

🤖 UTILISATION DES MODÈLES:
   Baseline: 9 (9.0%)
   V3: 91 (91.0%)
   Accord: 0/100 (0.0%)

❌ ERREURS (premières 10):
   1. CBSD → CGM (via v3)
   2. CBB → Unknown (via baseline)
   3. CMD → CGM (via v3)
   4. CBSD → CGM (via v3)
   5. CMD → CGM (via v3)
   6. CMD → CGM (via v3)
   7. CBSD → CGM (via v3)
   8. CBB → Unknown (via baseline)
   9. CBB → Unknown (via baseline)
   10. CBSD → CGM (via v3)



## 11. Comparaison avec Baseline et V3

In [11]:
print("="*70)
print("COMPARAISON: Baseline vs V3 vs Ensemble")
print("="*70)

print(f"\n{'Métrique':<15} {'Baseline':<12} {'V3':<12} {'Ensemble':<12} {'Meilleur'}")
print("-"*70)

# Overall
baseline_overall = 69.83
v3_overall = 65.05
ensemble_overall = overall_acc * 100
best_overall = max(baseline_overall, v3_overall, ensemble_overall)
print(f"{'Overall':<15} {baseline_overall:<12.2f} {v3_overall:<12.2f} {ensemble_overall:<12.2f} {'✅' if ensemble_overall == best_overall else ''}")

# Par classe (résultats connus)
comparisons = [
    ('CBB', 0.64, 34.0),
    ('CBSD', 48.64, 2.9),
    ('CGM', 0.0, 81.4),
    ('CMD', 97.07, 89.4),
    ('Healthy', 45.77, 40.3)
]

for class_abbrev, baseline_acc, v3_acc in comparisons:
    # Trouver la classe complète
    full_class = next((c for c in CLASS_NAMES if class_abbrev in c), class_abbrev)
    
    if full_class in class_metrics:
        ensemble_acc = class_metrics[full_class]['accuracy'] * 100
    else:
        ensemble_acc = 0.0
    
    best = max(baseline_acc, v3_acc, ensemble_acc)
    marker = '✅' if ensemble_acc == best else ''
    print(f"{class_abbrev:<15} {baseline_acc:<12.2f} {v3_acc:<12.2f} {ensemble_acc:<12.2f} {marker}")

print("\n" + "="*70)

# Critères de succès
print("\nCRITÈRES DE SUCCÈS:")
if ensemble_overall >= 75:
    print("✅ Overall ≥75%: EXCELLENT! Prêt pour déploiement")
elif ensemble_overall >= 72:
    print("✅ Overall ≥72%: SUCCÈS! Prêt pour déploiement")
elif ensemble_overall >= 70:
    print("⚠️ Overall ≥70%: Acceptable, mais peut être amélioré")
else:
    print("❌ Overall <70%: Considérer Option A (réentraînement)")

print("="*70)

COMPARAISON: Baseline vs V3 vs Ensemble

Métrique        Baseline     V3           Ensemble     Meilleur
----------------------------------------------------------------------
Overall         69.83        65.05        50.00        
CBB             0.64         34.00        0.00         
CBSD            48.64        2.90         0.00         
CGM             0.00         81.40        75.00        
CMD             97.07        89.40        78.57        
Healthy         45.77        40.30        0.00         


CRITÈRES DE SUCCÈS:
❌ Overall <70%: Considérer Option A (réentraînement)


## 12. Décision: Test Complet ou Arrêt?

In [12]:
# Décider si on continue avec le test complet
print("DÉCISION:")
print("="*70)

if ensemble_overall >= 70:
    print(f"✅ Test rapide réussi ({ensemble_overall:.2f}% ≥ 70%)")
    print(f"\n👉 RECOMMANDATION: Lancer le test complet (3,619 échantillons)")
    print(f"   - Temps estimé: ~40-60 minutes")
    print(f"   - Exécutez la section suivante (Section 13)")
else:
    print(f"⚠️ Test rapide sous les attentes ({ensemble_overall:.2f}% < 70%)")
    print(f"\n👉 RECOMMANDATION: Analyser les erreurs avant le test complet")
    print(f"   - Vérifiez les erreurs ci-dessus")
    print(f"   - Considérez Option A (ajuster class weights et réentraîner)")

print("="*70)

DÉCISION:
⚠️ Test rapide sous les attentes (54.00% < 70%)

👉 RECOMMANDATION: Analyser les erreurs avant le test complet
   - Vérifiez les erreurs ci-dessus
   - Considérez Option A (ajuster class weights et réentraîner)


## 13. Test Complet (3,619 échantillons) - OPTIONNEL

⚠️ **Attention**: Cette section prend ~40-60 minutes à exécuter!

Ne lancez cette section QUE si le test rapide est satisfaisant (≥70%).

In [ ]:
# DÉCOMMENTER POUR LANCER LE TEST COMPLET
# RUN_FULL_TEST = True
RUN_FULL_TEST = False  # Sécurité: désactivé par défaut

if RUN_FULL_TEST:
    print("="*70)
    print("TEST COMPLET - 3,619 ÉCHANTILLONS")
    print("="*70)
    print(f"⏱️ Temps estimé: ~45 minutes\n")
    
    full_results = []
    full_errors = []
    
    for sample in tqdm(test_data, desc="Prédictions complètes"):
        gt_class = extract_gt_class(sample)
        image_path = extract_image_path(sample)
        
        if not image_path or gt_class == 'Unknown':
            continue
        
        try:
            image = Image.open(image_path).convert('RGB')
            prediction = ensemble_predict(image)
            is_correct = prediction['predicted_class'] == gt_class
            
            full_results.append({
                'id': sample.get('id', 'unknown'),
                'ground_truth': gt_class,
                'predicted': prediction['predicted_class'],
                'confidence': prediction['confidence'],
                'source': prediction['source_model'],
                'correct': is_correct
            })
            
            if not is_correct:
                full_errors.append({
                    'id': sample.get('id', 'unknown'),
                    'true': gt_class,
                    'predicted': prediction['predicted_class'],
                    'source': prediction['source_model']
                })
        
        except Exception as e:
            print(f"\nErreur: {e}")
            continue
    
    # Sauvegarder les résultats
    results_file = BASE_DIR / "checkpoints" / "ensemble_full_results.json"
    results_file.parent.mkdir(parents=True, exist_ok=True)
    
    full_total = len(full_results)
    full_correct = sum(1 for r in full_results if r['correct'])
    full_acc = full_correct / full_total if full_total > 0 else 0
    
    # Métriques par classe
    full_class_metrics = {}
    for class_name in CLASS_NAMES:
        class_results = [r for r in full_results if r['ground_truth'] == class_name]
        if class_results:
            class_correct = sum(1 for r in class_results if r['correct'])
            full_class_metrics[class_name] = {
                'total': len(class_results),
                'correct': class_correct,
                'accuracy': class_correct / len(class_results)
            }
    
    results_data = {
        'overall_accuracy': full_acc,
        'total_samples': full_total,
        'correct': full_correct,
        'class_metrics': full_class_metrics,
        'errors_count': len(full_errors)
    }
    
    with open(results_file, 'w') as f:
        json.dump(results_data, f, indent=2)
    
    print(f"\n✅ Test complet terminé")
    print(f"\n📊 RÉSULTATS FINAUX:")
    print(f"   Overall: {full_acc*100:.2f}% ({full_correct}/{full_total})")
    print(f"\n📈 Par classe:")
    for class_name, metrics in full_class_metrics.items():
        abbrev = class_name.split('(')[1].strip(')') if '(' in class_name else class_name
        print(f"   {abbrev}: {metrics['accuracy']*100:.1f}% ({metrics['correct']}/{metrics['total']})")
    
    print(f"\n✅ Résultats sauvegardés: {results_file}")
    print("="*70)
else:
    print("⏭️ Test complet ignoré (RUN_FULL_TEST=False)")
    print("   Changez RUN_FULL_TEST = True pour lancer le test complet")

## 14. Conclusion et Prochaines Étapes

In [ ]:
print("="*70)
print("CONCLUSION")
print("="*70)

print(f"\n📊 Résumé des résultats (test rapide 100 échantillons):")
print(f"   Overall: {overall_acc*100:.2f}%")
print(f"\nComparaison:")
print(f"   Baseline: 69.83%")
print(f"   V3: 65.05%")
print(f"   Ensemble: {overall_acc*100:.2f}%")

if overall_acc >= 0.75:
    print(f"\n🎉 SUCCÈS EXCELLENT!")
    print(f"   L'ensemble dépasse les attentes (≥75%)")
    print(f"\n👉 PROCHAINES ÉTAPES:")
    print(f"   1. ✅ Lancer le test complet (Section 13)")
    print(f"   2. ✅ Déployer l'ensemble en production")
    print(f"   3. Documenter l'architecture à 2 modèles")
    print(f"   4. Optimiser l'inférence (batching, caching)")
elif overall_acc >= 0.72:
    print(f"\n✅ SUCCÈS!")
    print(f"   L'ensemble atteint l'objectif minimum (≥72%)")
    print(f"\n👉 PROCHAINES ÉTAPES:")
    print(f"   1. ✅ Lancer le test complet (Section 13)")
    print(f"   2. ✅ Déployer l'ensemble")
    print(f"   3. En parallèle: tester Option A (ajuster weights) pour améliorer")
elif overall_acc >= 0.70:
    print(f"\n⚠️ Succès partiel")
    print(f"   Meilleur que V3 mais légèrement sous baseline")
    print(f"\n👉 PROCHAINES ÉTAPES:")
    print(f"   1. Analyser les erreurs en détail")
    print(f"   2. Considérer améliorer la logique de routage")
    print(f"   3. Ou: essayer Option A (retrain avec weights ajustés)")
else:
    print(f"\n❌ Échec")
    print(f"   L'ensemble n'améliore pas les résultats")
    print(f"\n👉 PROCHAINES ÉTAPES:")
    print(f"   1. Option A: Réentraîner V3 avec class weights ajustés")
    print(f"      - CBSD=2.5, CGM=2.0 (au lieu de 1.5 et 3.0)")
    print(f"   2. Analyser pourquoi le routage ne fonctionne pas")

print("\n" + "="*70)
print("FIN DU NOTEBOOK")
print("="*70)